# 18uB — Model-specific freezes and admissible selection support

This stage converts the 18uA date partition into estimator-specific
training, validation, selection, holdout and external-test sets.

Five estimator scopes are frozen:

- one pooled all-rule estimator;
- four separate decision-rule estimators.

For each development fold, the fitted training set contains only
residual observations dated before the first validation date and
whose HKO outcome was available by the earliest applicable
validation decision time. Hence

\[
\max_{u\in T_k} u < \min_{d\in V_k} d,
\qquad
\max_{u\in T_k}\tau^{\mathrm{HKO},*}(u)
\leq \min_{i:d_i\in V_k}\tau_i .
\]

The model-specific holdout freeze is the earliest applicable
decision time in the 22–31 May holdout. Out-of-fold model and
calibrator selection may use only development labels available by
that freeze. Final fitting uses only pre-holdout residuals admitted
by the same freeze.

The primary June external evaluation transfers the same
pre-holdout fitted estimator without refitting on May holdout
outcomes. This preserves complete separation of the locked
holdout and external test from fitting and selection.

This stage freezes support only. It does not choose a GP kernel,
tree model, feature family, score hierarchy, calibrator or trading
threshold.

**Revision v2.** The model-specific selection stage reads the canonical current-label availability timestamp directly from 18uA, preventing duplicate merge suffixes.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run this notebook from the repository root, not {ROOT}"
    )

UTC = timezone.utc
STEP = "18uB"
MIN_TRAINING_DATES = 16
MIN_CALIBRATION_DATES = 8
CONTRACTS_PER_BOOK = 11

RULES = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]
RULE_ORDER = {
    rule: position
    for position, rule in enumerate(RULES)
}

U_DIR = (
    ROOT
    / "data/processed/18uA_chronological_partition_and_folds"
)
T_A_DIR = (
    ROOT
    / "data/processed/18tA_hko_publication_availability"
)
T_B_DIR = (
    ROOT
    / "data/processed/18tB_admissible_historical_information_sets"
)

ASSIGNMENT_PATH = U_DIR / "18uA_date_rule_assignment.csv"
DATE_PARTITION_PATH = U_DIR / "18uA_date_partition.csv"
FOLD_SUMMARY_PATH = U_DIR / "18uA_fold_summary.csv"
U_SUMMARY_PATH = U_DIR / "18uA_summary.json"
U_MANIFEST_PATH = U_DIR / "18uA_sha256_manifest.csv"

AVAILABILITY_PATH = (
    T_A_DIR / "18tA_hko_publication_availability_panel.csv"
)
RESIDUAL_PATH = (
    T_B_DIR / "18tB_residual_observation_panel.csv"
)
SAME_RULE_PAIRS_PATH = (
    T_B_DIR / "18tB_admissible_history_pairs_same_rule.csv"
)
T_B_SUMMARY_PATH = T_B_DIR / "18tB_summary.json"

OUT_DIR = (
    ROOT
    / "data/processed/18uB_model_specific_freeze_support"
)
REPORT_DIR = (
    ROOT
    / "reports/18uB_model_specific_freeze_support"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = [
    ASSIGNMENT_PATH,
    DATE_PARTITION_PATH,
    FOLD_SUMMARY_PATH,
    U_SUMMARY_PATH,
    U_MANIFEST_PATH,
    AVAILABILITY_PATH,
    RESIDUAL_PATH,
    SAME_RULE_PAIRS_PATH,
    T_B_SUMMARY_PATH,
]

for path in REQUIRED_INPUTS:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified input is missing: {path}"
        )

EXPECTED_SCOPE_SUMMARY = {
    "pooled_all_rules": {
        "freeze_utc": "2026-05-20 16:00:00+00:00",
        "oof_rows": 144,
        "oof_dates": 38,
        "selection_rows": 136,
        "selection_dates": 36,
        "fit_rows": 208,
        "fit_dates": 60,
        "holdout_rows": 40,
        "holdout_dates": 10,
        "external_rows": 119,
        "external_dates": 30,
    },
    "rule_specific_24h_prior": {
        "freeze_utc": "2026-05-20 16:00:00+00:00",
        "oof_rows": 38,
        "oof_dates": 38,
        "selection_rows": 36,
        "selection_dates": 36,
        "fit_rows": 54,
        "fit_dates": 54,
        "holdout_rows": 10,
        "holdout_dates": 10,
        "external_rows": 30,
        "external_dates": 30,
    },
    "rule_specific_12h_prior": {
        "freeze_utc": "2026-05-21 04:00:00+00:00",
        "oof_rows": 35,
        "oof_dates": 35,
        "selection_rows": 33,
        "selection_dates": 33,
        "fit_rows": 51,
        "fit_dates": 51,
        "holdout_rows": 10,
        "holdout_dates": 10,
        "external_rows": 30,
        "external_dates": 30,
    },
    "rule_specific_6h_prior": {
        "freeze_utc": "2026-05-21 10:00:00+00:00",
        "oof_rows": 35,
        "oof_dates": 35,
        "selection_rows": 34,
        "selection_dates": 34,
        "fit_rows": 51,
        "fit_dates": 51,
        "holdout_rows": 10,
        "holdout_dates": 10,
        "external_rows": 29,
        "external_dates": 29,
    },
    "rule_specific_event_day_open": {
        "freeze_utc": "2026-05-21 16:00:00+00:00",
        "oof_rows": 36,
        "oof_dates": 36,
        "selection_rows": 35,
        "selection_dates": 35,
        "fit_rows": 54,
        "fit_dates": 54,
        "holdout_rows": 10,
        "holdout_dates": 10,
        "external_rows": 30,
        "external_dates": 30,
    },
}

EXPECTED_FOLD_TRAINING_ROWS = {
    ("pooled_all_rules", 1): (56, 22),
    ("pooled_all_rules", 2): (96, 32),
    ("pooled_all_rules", 3): (136, 42),
    ("pooled_all_rules", 4): (168, 50),
    ("rule_specific_24h_prior", 1): (16, 16),
    ("rule_specific_24h_prior", 2): (26, 26),
    ("rule_specific_24h_prior", 3): (36, 36),
    ("rule_specific_24h_prior", 4): (44, 44),
    ("rule_specific_12h_prior", 1): (16, 16),
    ("rule_specific_12h_prior", 2): (23, 23),
    ("rule_specific_12h_prior", 3): (33, 33),
    ("rule_specific_12h_prior", 4): (41, 41),
    ("rule_specific_6h_prior", 1): (16, 16),
    ("rule_specific_6h_prior", 2): (23, 23),
    ("rule_specific_6h_prior", 3): (32, 32),
    ("rule_specific_6h_prior", 4): (40, 40),
    ("rule_specific_event_day_open", 1): (18, 18),
    ("rule_specific_event_day_open", 2): (26, 26),
    ("rule_specific_event_day_open", 3): (35, 35),
    ("rule_specific_event_day_open", 4): (43, 43),
}

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(
    series: pd.Series,
    *,
    name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean column {name}: {bad}"
        )

    return parsed.astype(bool)


def verify_manifest(path: Path) -> None:
    manifest = pd.read_csv(path)
    failures: list[str] = []

    for row in manifest.itertuples(index=False):
        candidate = ROOT / row.path

        if not candidate.is_file():
            failures.append(f"MISSING: {row.path}")
            continue

        digest = sha256_file(candidate)
        if digest != row.sha256:
            failures.append(f"HASH: {row.path}")

        if candidate.stat().st_size != int(row.size_bytes):
            failures.append(f"SIZE: {row.path}")

    if failures:
        raise AssertionError(
            f"Manifest verification failed for {path}:\n"
            + "\n".join(failures)
        )


def scope_rule(scope_id: str) -> str | None:
    prefix = "rule_specific_"
    if scope_id == "pooled_all_rules":
        return None
    if scope_id.startswith(prefix):
        return scope_id[len(prefix):]
    raise ValueError(f"Unknown estimator scope: {scope_id}")


def date_balanced_weights(
    frame: pd.DataFrame,
    date_column: str,
) -> pd.Series:
    if frame.empty:
        return pd.Series(
            dtype=float,
            index=frame.index,
        )

    n_dates = frame[date_column].nunique()
    rows_per_date = frame.groupby(
        date_column
    )[date_column].transform("size")

    weights = 1.0 / (
        float(n_dates)
        * rows_per_date.astype(float)
    )

    date_totals = weights.groupby(
        frame[date_column]
    ).sum()

    if not np.isclose(
        date_totals.to_numpy(dtype=float),
        1.0 / float(n_dates),
        rtol=0.0,
        atol=1e-12,
    ).all():
        raise AssertionError(
            "Date-balanced weights do not assign equal "
            "total weight to each date."
        )

    if not np.isclose(
        weights.sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):
        raise AssertionError(
            "Date-balanced weights do not sum to one."
        )

    return weights


verify_manifest(U_MANIFEST_PATH)

with U_SUMMARY_PATH.open(encoding="utf-8") as handle:
    u_summary = json.load(handle)
with T_B_SUMMARY_PATH.open(encoding="utf-8") as handle:
    t_summary = json.load(handle)

if u_summary.get("verdict") != "PASS":
    raise AssertionError("18uA is not a PASS release.")
if t_summary.get("verdict") != "PASS":
    raise AssertionError("18tB is not a PASS release.")

assignment = pd.read_csv(
    ASSIGNMENT_PATH,
    low_memory=False,
)
date_partition = pd.read_csv(
    DATE_PARTITION_PATH,
    low_memory=False,
)
fold_summary_input = pd.read_csv(
    FOLD_SUMMARY_PATH,
    low_memory=False,
)
availability = pd.read_csv(
    AVAILABILITY_PATH,
    low_memory=False,
)
residuals = pd.read_csv(
    RESIDUAL_PATH,
    low_memory=False,
)
same_rule_pairs = pd.read_csv(
    SAME_RULE_PAIRS_PATH,
    low_memory=False,
)

assignment["event_date"] = pd.to_datetime(
    assignment["event_date"],
    errors="raise",
)
assignment["decision_cutoff_utc"] = pd.to_datetime(
    assignment["decision_cutoff_utc"],
    utc=True,
    errors="raise",
)
assignment["model_ready"] = parse_bool(
    assignment["model_ready"],
    name="assignment.model_ready",
)
assignment["current_prediction_eligible"] = parse_bool(
    assignment["current_prediction_eligible"],
    name="assignment.current_prediction_eligible",
)
assignment["development_fold"] = pd.to_numeric(
    assignment["development_fold"],
    errors="coerce",
).astype("Int64")
assignment[
    "current_label_available_utc"
] = pd.to_datetime(
    assignment["current_label_available_utc"],
    utc=True,
    errors="raise",
)

date_partition["event_date"] = pd.to_datetime(
    date_partition["event_date"],
    errors="raise",
)
fold_summary_input[
    "validation_start_date"
] = pd.to_datetime(
    fold_summary_input[
        "validation_start_date"
    ],
    errors="raise",
)
fold_summary_input[
    "validation_end_date"
] = pd.to_datetime(
    fold_summary_input[
        "validation_end_date"
    ],
    errors="raise",
)

availability["event_date"] = pd.to_datetime(
    availability["event_date"],
    errors="raise",
)
availability[
    "hko_publication_available_utc"
] = pd.to_datetime(
    availability[
        "hko_publication_available_utc"
    ],
    utc=True,
    errors="raise",
)

residuals["event_date"] = pd.to_datetime(
    residuals["event_date"],
    errors="raise",
)
residuals["decision_cutoff_utc"] = pd.to_datetime(
    residuals["decision_cutoff_utc"],
    utc=True,
    errors="raise",
)
residuals[
    "selected_run_available_utc"
] = pd.to_datetime(
    residuals[
        "selected_run_available_utc"
    ],
    utc=True,
    errors="raise",
)
residuals[
    "hko_publication_available_utc"
] = pd.to_datetime(
    residuals[
        "hko_publication_available_utc"
    ],
    utc=True,
    errors="raise",
)
residuals[
    "historical_weather_admissible_at_own_decision"
] = parse_bool(
    residuals[
        "historical_weather_admissible_at_own_decision"
    ],
    name=(
        "residuals."
        "historical_weather_admissible_at_own_decision"
    ),
)

same_rule_pairs["current_event_date"] = pd.to_datetime(
    same_rule_pairs["current_event_date"],
    errors="raise",
)
same_rule_pairs["history_event_date"] = pd.to_datetime(
    same_rule_pairs["history_event_date"],
    errors="raise",
)
same_rule_pairs[
    "current_decision_cutoff_utc"
] = pd.to_datetime(
    same_rule_pairs[
        "current_decision_cutoff_utc"
    ],
    utc=True,
    errors="raise",
)
same_rule_pairs[
    "history_hko_available_utc"
] = pd.to_datetime(
    same_rule_pairs[
        "history_hko_available_utc"
    ],
    utc=True,
    errors="raise",
)

if len(assignment) != 412:
    raise AssertionError(
        f"Expected 412 18uA rows, found {len(assignment)}"
    )
if len(residuals) != 375:
    raise AssertionError(
        f"Expected 375 residual rows, found {len(residuals)}"
    )
if len(same_rule_pairs) != 16638:
    raise AssertionError(
        f"Expected 16,638 same-rule pairs, "
        f"found {len(same_rule_pairs)}"
    )

print("Verified 18uA and 18t inputs: PASS")

Verified 18uA and 18t inputs: PASS


In [3]:
scope_rows = [
    {
        "scope_id": "pooled_all_rules",
        "scope_type": "POOLED",
        "applicable_decision_rule": "",
        "decision_rule_order": -1,
    }
]

for rule in RULES:
    scope_rows.append(
        {
            "scope_id": f"rule_specific_{rule}",
            "scope_type": "RULE_SPECIFIC",
            "applicable_decision_rule": rule,
            "decision_rule_order": RULE_ORDER[rule],
        }
    )

scope_registry = pd.DataFrame(scope_rows)
scope_registry[
    "minimum_training_dates"
] = MIN_TRAINING_DATES
scope_registry[
    "minimum_calibration_dates"
] = MIN_CALIBRATION_DATES
scope_registry[
    "holdout_used_for_selection"
] = False
scope_registry[
    "external_test_used_for_selection"
] = False
scope_registry[
    "external_primary_refit_on_holdout_labels"
] = False
scope_registry[
    "external_primary_protocol"
] = (
    "transfer identical pre-holdout fitted estimator "
    "to June without refit"
)
scope_registry[
    "score_hierarchy_selected"
] = False
scope_registry[
    "model_specification_selected"
] = False

development_ready = assignment.loc[
    assignment["evaluation_block"].eq(
        "DEVELOPMENT"
    )
    & assignment["model_ready"]
].copy()

holdout_ready = assignment.loc[
    assignment["evaluation_block"].eq(
        "INTERNAL_HOLDOUT"
    )
    & assignment["model_ready"]
].copy()

external_ready = assignment.loc[
    assignment["evaluation_block"].eq(
        "EXTERNAL_TEST"
    )
    & assignment["model_ready"]
].copy()

if len(development_ready) != 144:
    raise AssertionError(
        f"Development-ready rows: {len(development_ready)}"
    )
if len(holdout_ready) != 40:
    raise AssertionError(
        f"Holdout-ready rows: {len(holdout_ready)}"
    )
if len(external_ready) != 119:
    raise AssertionError(
        f"External-ready rows: {len(external_ready)}"
    )

print("Estimator-scope registry and evaluation supports: PASS")
display(scope_registry)

Estimator-scope registry and evaluation supports: PASS


,scope_id,scope_type,applicable_decision_rule,decision_rule_order,minimum_training_dates,minimum_calibration_dates,holdout_used_for_selection,external_test_used_for_selection,external_primary_refit_on_holdout_labels,external_primary_protocol,score_hierarchy_selected,model_specification_selected
0,pooled_all_rules,POOLED,,-1,16,8,False,False,False,transfer identical pre-holdout fitted estimato...,False,False
1,rule_specific_24h_prior,RULE_SPECIFIC,24h_prior,0,16,8,False,False,False,transfer identical pre-holdout fitted estimato...,False,False
2,rule_specific_12h_prior,RULE_SPECIFIC,12h_prior,1,16,8,False,False,False,transfer identical pre-holdout fitted estimato...,False,False
3,rule_specific_6h_prior,RULE_SPECIFIC,6h_prior,2,16,8,False,False,False,transfer identical pre-holdout fitted estimato...,False,False
4,rule_specific_event_day_open,RULE_SPECIFIC,event_day_open,3,16,8,False,False,False,transfer identical pre-holdout fitted estimato...,False,False


In [4]:
fold_plan_rows: list[dict[str, Any]] = []
fold_training_membership_frames: list[pd.DataFrame] = []
fold_validation_membership_frames: list[pd.DataFrame] = []

for scope in scope_registry.itertuples(index=False):
    rule = scope_rule(scope.scope_id)

    for fold_id in [1, 2, 3, 4]:
        validation = development_ready.loc[
            development_ready[
                "development_fold"
            ].eq(fold_id)
        ].copy()

        if rule is not None:
            validation = validation.loc[
                validation["decision_rule"].eq(rule)
            ].copy()

        if validation.empty:
            raise AssertionError(
                f"No validation rows for {scope.scope_id}, "
                f"fold {fold_id}"
            )

        earliest_validation_date = validation[
            "event_date"
        ].min()
        latest_validation_date = validation[
            "event_date"
        ].max()
        fold_freeze_utc = validation[
            "decision_cutoff_utc"
        ].min()

        training = residuals.loc[
            residuals["event_date"].lt(
                earliest_validation_date
            )
            & residuals[
                "hko_publication_available_utc"
            ].le(fold_freeze_utc)
            & residuals[
                "selected_run_available_utc"
            ].le(
                residuals["decision_cutoff_utc"]
            )
        ].copy()

        if rule is not None:
            training = training.loc[
                training["decision_rule"].eq(rule)
            ].copy()

        training_dates = training[
            "event_date"
        ].nunique()

        if training_dates < MIN_TRAINING_DATES:
            raise AssertionError(
                f"{scope.scope_id}, fold {fold_id}: "
                f"only {training_dates} training dates"
            )

        if not (
            training["event_date"]
            < earliest_validation_date
        ).all():
            raise AssertionError(
                "A fold training date is not strictly "
                "before the first validation date."
            )

        if not (
            training[
                "hko_publication_available_utc"
            ]
            <= fold_freeze_utc
        ).all():
            raise AssertionError(
                "A fold training label was unavailable by "
                "the earliest validation decision."
            )

        if not training[
            "historical_weather_admissible_at_own_decision"
        ].all():
            raise AssertionError(
                "A fold training weather run was not "
                "admissible at its own decision."
            )

        expected_rows, expected_dates = (
            EXPECTED_FOLD_TRAINING_ROWS[
                (scope.scope_id, fold_id)
            ]
        )

        if (
            len(training) != expected_rows
            or training_dates != expected_dates
        ):
            raise AssertionError(
                f"Unexpected fold training support for "
                f"{scope.scope_id}, fold {fold_id}: "
                f"expected rows/dates "
                f"{expected_rows}/{expected_dates}, "
                f"found {len(training)}/{training_dates}"
            )

        training = training.sort_values(
            [
                "event_date",
                "decision_rule_order",
            ]
        ).reset_index(drop=True)
        training[
            "date_balanced_training_weight"
        ] = date_balanced_weights(
            training,
            "event_date",
        )
        training["scope_id"] = scope.scope_id
        training["development_fold"] = fold_id
        training["fold_freeze_utc"] = fold_freeze_utc
        training[
            "earliest_validation_date"
        ] = earliest_validation_date
        training[
            "latest_validation_date"
        ] = latest_validation_date

        validation = validation.sort_values(
            [
                "event_date",
                "decision_rule_order",
            ]
        ).reset_index(drop=True)
        validation[
            "date_balanced_validation_weight"
        ] = date_balanced_weights(
            validation,
            "event_date",
        )
        validation["scope_id"] = scope.scope_id
        validation[
            "fold_freeze_utc"
        ] = fold_freeze_utc

        fold_training_membership_frames.append(
            training
        )
        fold_validation_membership_frames.append(
            validation
        )

        fold_plan_rows.append(
            {
                "scope_id": scope.scope_id,
                "scope_type": scope.scope_type,
                "applicable_decision_rule": (
                    scope.applicable_decision_rule
                ),
                "development_fold": fold_id,
                "earliest_validation_date": (
                    earliest_validation_date
                ),
                "latest_validation_date": (
                    latest_validation_date
                ),
                "fold_freeze_utc": fold_freeze_utc,
                "training_rows": len(training),
                "training_dates": training_dates,
                "latest_training_date": (
                    training["event_date"].max()
                ),
                "latest_training_label_available_utc": (
                    training[
                        "hko_publication_available_utc"
                    ].max()
                ),
                "validation_rows": len(validation),
                "validation_dates": validation[
                    "event_date"
                ].nunique(),
                "training_date_condition_passed": True,
                "training_label_availability_condition_passed": True,
                "minimum_training_dates_passed": True,
            }
        )

fold_plan = pd.DataFrame(fold_plan_rows)
fold_training_membership = pd.concat(
    fold_training_membership_frames,
    ignore_index=True,
)
fold_validation_membership = pd.concat(
    fold_validation_membership_frames,
    ignore_index=True,
)

if len(fold_plan) != 20:
    raise AssertionError(
        f"Expected 20 scope-fold rows, found {len(fold_plan)}"
    )

print("Expanding-fold training and validation support: PASS")
display(fold_plan)

Expanding-fold training and validation support: PASS


,scope_id,scope_type,applicable_decision_rule,development_fold,earliest_validation_date,latest_validation_date,fold_freeze_utc,training_rows,training_dates,latest_training_date,latest_training_label_available_utc,validation_rows,validation_dates,training_date_condition_passed,training_label_availability_condition_passed,minimum_training_dates_passed
0,pooled_all_rules,POOLED,,1,2026-04-12,2026-04-21,2026-04-10 16:00:00+00:00,56,22,2026-04-09,2026-04-10 06:00:00+00:00,32,10,True,True,True
1,pooled_all_rules,POOLED,,2,2026-04-22,2026-05-01,2026-04-20 16:00:00+00:00,96,32,2026-04-19,2026-04-20 06:00:00+00:00,40,10,True,True,True
2,pooled_all_rules,POOLED,,3,2026-05-02,2026-05-10,2026-04-30 16:00:00+00:00,136,42,2026-04-29,2026-04-30 06:00:00+00:00,36,9,True,True,True
3,pooled_all_rules,POOLED,,4,2026-05-11,2026-05-21,2026-05-09 16:00:00+00:00,168,50,2026-05-07,2026-05-08 06:00:00+00:00,36,9,True,True,True
4,rule_specific_24h_prior,RULE_SPECIFIC,24h_prior,1,2026-04-12,2026-04-21,2026-04-10 16:00:00+00:00,16,16,2026-04-09,2026-04-10 06:00:00+00:00,10,10,True,True,True
5,rule_specific_24h_prior,RULE_SPECIFIC,24h_prior,2,2026-04-22,2026-05-01,2026-04-20 16:00:00+00:00,26,26,2026-04-19,2026-04-20 06:00:00+00:00,10,10,True,True,True
6,rule_specific_24h_prior,RULE_SPECIFIC,24h_prior,3,2026-05-02,2026-05-10,2026-04-30 16:00:00+00:00,36,36,2026-04-29,2026-04-30 06:00:00+00:00,9,9,True,True,True
7,rule_specific_24h_prior,RULE_SPECIFIC,24h_prior,4,2026-05-11,2026-05-21,2026-05-09 16:00:00+00:00,44,44,2026-05-07,2026-05-08 06:00:00+00:00,9,9,True,True,True
8,rule_specific_12h_prior,RULE_SPECIFIC,12h_prior,1,2026-04-15,2026-04-21,2026-04-14 04:00:00+00:00,16,16,2026-04-12,2026-04-13 06:00:00+00:00,7,7,True,True,True
9,rule_specific_12h_prior,RULE_SPECIFIC,12h_prior,2,2026-04-22,2026-05-01,2026-04-21 04:00:00+00:00,23,23,2026-04-19,2026-04-20 06:00:00+00:00,10,10,True,True,True


In [5]:
freeze_summary_rows: list[dict[str, Any]] = []
oof_selection_frames: list[pd.DataFrame] = []
final_fit_frames: list[pd.DataFrame] = []
evaluation_frames: list[pd.DataFrame] = []

for scope in scope_registry.itertuples(index=False):
    rule = scope_rule(scope.scope_id)

    holdout = holdout_ready.copy()
    external = external_ready.copy()
    oof = development_ready.copy()

    if rule is not None:
        holdout = holdout.loc[
            holdout["decision_rule"].eq(rule)
        ].copy()
        external = external.loc[
            external["decision_rule"].eq(rule)
        ].copy()
        oof = oof.loc[
            oof["decision_rule"].eq(rule)
        ].copy()

    if holdout.empty:
        raise AssertionError(
            f"No holdout rows for {scope.scope_id}"
        )

    holdout_freeze_utc = holdout[
        "decision_cutoff_utc"
    ].min()

    oof[
        "freeze_admissible_for_selection"
    ] = oof[
        "current_label_available_utc"
    ].le(holdout_freeze_utc)
    oof["scope_id"] = scope.scope_id
    oof[
        "model_specific_holdout_freeze_utc"
    ] = holdout_freeze_utc
    oof[
        "common_candidate_oof_support_intersection_pending"
    ] = True
    oof[
        "selected_model_specification"
    ] = False

    selection = oof.loc[
        oof[
            "freeze_admissible_for_selection"
        ]
    ].copy()
    selection[
        "date_balanced_selection_weight"
    ] = date_balanced_weights(
        selection,
        "event_date",
    )

    fit = residuals.loc[
        residuals["event_date"].le(
            pd.Timestamp("2026-05-21")
        )
        & residuals[
            "hko_publication_available_utc"
        ].le(holdout_freeze_utc)
        & residuals[
            "selected_run_available_utc"
        ].le(
            residuals["decision_cutoff_utc"]
        )
    ].copy()

    if rule is not None:
        fit = fit.loc[
            fit["decision_rule"].eq(rule)
        ].copy()

    if fit["event_date"].ge(
        pd.Timestamp("2026-05-22")
    ).any():
        raise AssertionError(
            "A holdout or later label entered final fitting."
        )

    fit = fit.sort_values(
        [
            "event_date",
            "decision_rule_order",
        ]
    ).reset_index(drop=True)
    fit["date_balanced_fit_weight"] = (
        date_balanced_weights(
            fit,
            "event_date",
        )
    )
    fit["scope_id"] = scope.scope_id
    fit[
        "model_specific_holdout_freeze_utc"
    ] = holdout_freeze_utc
    fit[
        "contains_holdout_label"
    ] = False
    fit[
        "contains_external_label"
    ] = False

    holdout = holdout.copy()
    holdout["evaluation_stage"] = (
        "LOCKED_INTERNAL_HOLDOUT"
    )
    holdout[
        "date_balanced_evaluation_weight"
    ] = date_balanced_weights(
        holdout,
        "event_date",
    )
    holdout["scope_id"] = scope.scope_id
    holdout[
        "fitted_parameter_source"
    ] = "PRE_HOLDOUT_FREEZE"
    holdout[
        "refit_on_holdout_labels"
    ] = False

    external = external.copy()
    external["evaluation_stage"] = (
        "EXTERNAL_OOT_TRANSFER"
    )
    external[
        "date_balanced_evaluation_weight"
    ] = date_balanced_weights(
        external,
        "event_date",
    )
    external["scope_id"] = scope.scope_id
    external[
        "fitted_parameter_source"
    ] = "IDENTICAL_PRE_HOLDOUT_FIT"
    external[
        "refit_on_holdout_labels"
    ] = False

    expected = EXPECTED_SCOPE_SUMMARY[
        scope.scope_id
    ]
    actual = {
        "freeze_utc": str(holdout_freeze_utc),
        "oof_rows": len(oof),
        "oof_dates": oof["event_date"].nunique(),
        "selection_rows": len(selection),
        "selection_dates": selection[
            "event_date"
        ].nunique(),
        "fit_rows": len(fit),
        "fit_dates": fit["event_date"].nunique(),
        "holdout_rows": len(holdout),
        "holdout_dates": holdout[
            "event_date"
        ].nunique(),
        "external_rows": len(external),
        "external_dates": external[
            "event_date"
        ].nunique(),
    }

    if actual != expected:
        raise AssertionError(
            f"Scope totals differ for {scope.scope_id}:\n"
            f"Expected={expected}\nActual={actual}"
        )

    if selection[
        "event_date"
    ].nunique() < MIN_CALIBRATION_DATES:
        raise AssertionError(
            f"{scope.scope_id} has fewer than "
            f"{MIN_CALIBRATION_DATES} selection dates."
        )

    oof_selection_frames.append(oof)
    final_fit_frames.append(fit)
    evaluation_frames.extend(
        [holdout, external]
    )

    freeze_summary_rows.append(
        {
            "scope_id": scope.scope_id,
            "scope_type": scope.scope_type,
            "applicable_decision_rule": (
                scope.applicable_decision_rule
            ),
            "model_specific_holdout_freeze_utc": (
                holdout_freeze_utc
            ),
            "development_oof_rows": len(oof),
            "development_oof_dates": oof[
                "event_date"
            ].nunique(),
            "freeze_admissible_selection_rows": (
                len(selection)
            ),
            "freeze_admissible_selection_dates": (
                selection["event_date"].nunique()
            ),
            "latest_selection_label_date": (
                selection["event_date"].max()
            ),
            "final_fit_rows": len(fit),
            "final_fit_dates": fit[
                "event_date"
            ].nunique(),
            "latest_final_fit_label_date": (
                fit["event_date"].max()
            ),
            "internal_holdout_rows": len(holdout),
            "internal_holdout_dates": holdout[
                "event_date"
            ].nunique(),
            "external_test_rows": len(external),
            "external_test_dates": external[
                "event_date"
            ].nunique(),
            "minimum_calibration_dates": (
                MIN_CALIBRATION_DATES
            ),
            "minimum_calibration_dates_passed": True,
            "holdout_labels_used_for_fitting": False,
            "external_labels_used_for_fitting": False,
            "external_refit_on_holdout_labels": False,
        }
    )

oof_selection_support = pd.concat(
    oof_selection_frames,
    ignore_index=True,
)
final_fit_membership = pd.concat(
    final_fit_frames,
    ignore_index=True,
)
evaluation_support = pd.concat(
    evaluation_frames,
    ignore_index=True,
)
freeze_summary = pd.DataFrame(
    freeze_summary_rows
)

if len(freeze_summary) != 5:
    raise AssertionError(
        f"Expected five scope summaries, "
        f"found {len(freeze_summary)}"
    )

print("Model-specific freeze and selection support: PASS")
display(freeze_summary)

Model-specific freeze and selection support: PASS


,scope_id,scope_type,applicable_decision_rule,model_specific_holdout_freeze_utc,development_oof_rows,development_oof_dates,freeze_admissible_selection_rows,freeze_admissible_selection_dates,latest_selection_label_date,final_fit_rows,...,latest_final_fit_label_date,internal_holdout_rows,internal_holdout_dates,external_test_rows,external_test_dates,minimum_calibration_dates,minimum_calibration_dates_passed,holdout_labels_used_for_fitting,external_labels_used_for_fitting,external_refit_on_holdout_labels
0,pooled_all_rules,POOLED,,2026-05-20 16:00:00+00:00,144,38,136,36,2026-05-19,208,...,2026-05-19,40,10,119,30,8,True,False,False,False
1,rule_specific_24h_prior,RULE_SPECIFIC,24h_prior,2026-05-20 16:00:00+00:00,38,38,36,36,2026-05-19,54,...,2026-05-19,10,10,30,30,8,True,False,False,False
2,rule_specific_12h_prior,RULE_SPECIFIC,12h_prior,2026-05-21 04:00:00+00:00,35,35,33,33,2026-05-19,51,...,2026-05-19,10,10,30,30,8,True,False,False,False
3,rule_specific_6h_prior,RULE_SPECIFIC,6h_prior,2026-05-21 10:00:00+00:00,35,35,34,34,2026-05-20,51,...,2026-05-20,10,10,29,29,8,True,False,False,False
4,rule_specific_event_day_open,RULE_SPECIFIC,event_day_open,2026-05-21 16:00:00+00:00,36,36,35,35,2026-05-20,54,...,2026-05-20,10,10,30,30,8,True,False,False,False


In [6]:
check_rows: list[dict[str, Any]] = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "18uA_input_pass",
    u_summary.get("verdict") == "PASS",
    str(u_summary.get("verdict")),
)
add_check(
    "estimator_scopes_5",
    len(scope_registry) == 5,
    f"scopes={len(scope_registry)}",
)
add_check(
    "fold_scope_rows_20",
    len(fold_plan) == 20,
    f"rows={len(fold_plan)}",
)
add_check(
    "fold_training_dates_at_least_16",
    fold_plan[
        "training_dates"
    ].ge(MIN_TRAINING_DATES).all(),
    f"minimum={fold_plan['training_dates'].min()}",
)
add_check(
    "fold_training_dates_strictly_prior",
    (
        fold_plan["latest_training_date"]
        < fold_plan["earliest_validation_date"]
    ).all(),
    "max training date < min validation date",
)
add_check(
    "fold_training_labels_available_by_freeze",
    (
        fold_plan[
            "latest_training_label_available_utc"
        ]
        <= fold_plan["fold_freeze_utc"]
    ).all(),
    "publication-admissible fixed training blocks",
)
add_check(
    "development_oof_rows_144_pooled",
    int(
        freeze_summary.loc[
            freeze_summary["scope_id"].eq(
                "pooled_all_rules"
            ),
            "development_oof_rows",
        ].iloc[0]
    )
    == 144,
    "pooled temporal OOF envelope",
)
add_check(
    "holdout_rows_40_pooled",
    int(
        freeze_summary.loc[
            freeze_summary["scope_id"].eq(
                "pooled_all_rules"
            ),
            "internal_holdout_rows",
        ].iloc[0]
    )
    == 40,
    "440 contract cells",
)
add_check(
    "external_rows_119_pooled",
    int(
        freeze_summary.loc[
            freeze_summary["scope_id"].eq(
                "pooled_all_rules"
            ),
            "external_test_rows",
        ].iloc[0]
    )
    == 119,
    "1,309 contract cells",
)
add_check(
    "selection_support_freeze_admissible",
    oof_selection_support.loc[
        oof_selection_support[
            "freeze_admissible_for_selection"
        ]
    ][
        "current_label_available_utc"
    ].le(
        oof_selection_support.loc[
            oof_selection_support[
                "freeze_admissible_for_selection"
            ],
            "model_specific_holdout_freeze_utc",
        ]
    ).all(),
    "OOF labels admitted by model-specific freeze",
)
add_check(
    "final_fit_pre_holdout_only",
    final_fit_membership[
        "event_date"
    ].lt(pd.Timestamp("2026-05-22")).all(),
    "no holdout or external label",
)
add_check(
    "holdout_labels_not_used_for_fit",
    not final_fit_membership[
        "contains_holdout_label"
    ].any(),
    "locked holdout",
)
add_check(
    "external_labels_not_used_for_fit",
    not final_fit_membership[
        "contains_external_label"
    ].any(),
    "external OOT test",
)
add_check(
    "external_no_refit_on_holdout",
    not evaluation_support.loc[
        evaluation_support[
            "evaluation_stage"
        ].eq("EXTERNAL_OOT_TRANSFER"),
        "refit_on_holdout_labels",
    ].any(),
    "same pre-holdout fit transferred to June",
)
add_check(
    "minimum_calibration_dates_passed",
    freeze_summary[
        "freeze_admissible_selection_dates"
    ].ge(MIN_CALIBRATION_DATES).all(),
    (
        "minimum selection dates="
        f"{freeze_summary['freeze_admissible_selection_dates'].min()}"
    ),
)
add_check(
    "date_balanced_training_weights",
    np.isclose(
        fold_training_membership.groupby(
            ["scope_id", "development_fold"]
        )["date_balanced_training_weight"].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ).all(),
    "each scope-fold training weight sums to one",
)
add_check(
    "date_balanced_validation_weights",
    np.isclose(
        fold_validation_membership.groupby(
            ["scope_id", "development_fold"]
        )["date_balanced_validation_weight"].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ).all(),
    "each scope-fold validation weight sums to one",
)
add_check(
    "model_specification_not_selected",
    not scope_registry[
        "model_specification_selected"
    ].any(),
    "selection deferred",
)
add_check(
    "score_hierarchy_not_selected",
    not scope_registry[
        "score_hierarchy_selected"
    ].any(),
    "GP/tree evaluation criterion deferred",
)
add_check(
    "no_market_or_bridge_requirement",
    (
        not assignment[
            "market_information_required_for_split"
        ].any()
        and not assignment[
            "probability_bridge_required_for_split"
        ].any()
    ),
    "support design is model-agnostic",
)

integrity = pd.DataFrame(check_rows)
if not integrity["passed"].all():
    raise AssertionError(
        "18uB blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "scope_id",
        "development_fold",
        "event_date",
        "decision_rule",
        "detail",
        "blocking",
    ]
)

print("18uB integrity checks: PASS")

18uB integrity checks: PASS


In [7]:
output_frames = {
    "scope_registry": scope_registry,
    "fold_plan": fold_plan,
    "fold_training_membership": (
        fold_training_membership
    ),
    "fold_validation_membership": (
        fold_validation_membership
    ),
    "oof_selection_support": (
        oof_selection_support
    ),
    "final_fit_membership": (
        final_fit_membership
    ),
    "evaluation_support": evaluation_support,
    "freeze_summary": freeze_summary,
    "integrity": integrity,
    "issues": issues,
}

output_paths = {
    "scope_registry": (
        OUT_DIR / "18uB_estimator_scope_registry.csv"
    ),
    "fold_plan": (
        OUT_DIR / "18uB_fold_training_plan.csv"
    ),
    "fold_training_membership": (
        OUT_DIR
        / "18uB_fold_training_membership.csv"
    ),
    "fold_validation_membership": (
        OUT_DIR
        / "18uB_fold_validation_membership.csv"
    ),
    "oof_selection_support": (
        OUT_DIR
        / "18uB_oof_selection_support.csv"
    ),
    "final_fit_membership": (
        OUT_DIR
        / "18uB_final_fit_membership.csv"
    ),
    "evaluation_support": (
        OUT_DIR / "18uB_evaluation_support.csv"
    ),
    "freeze_summary": (
        OUT_DIR / "18uB_freeze_summary.csv"
    ),
    "integrity": (
        OUT_DIR / "18uB_integrity_checks.csv"
    ),
    "issues": OUT_DIR / "18uB_issues.csv",
}

for key, frame in output_frames.items():
    output = frame.copy()

    for column in output.columns:
        if "date" in column.lower():
            if pd.api.types.is_datetime64_any_dtype(
                output[column]
            ):
                output[column] = output[
                    column
                ].dt.strftime("%Y-%m-%d")

        if (
            "cutoff" in column.lower()
            or "freeze" in column.lower()
            or column.lower().endswith("_utc")
            or column.lower().endswith("_hkt")
            or "available" in column.lower()
            or "initialisation" in column.lower()
            or "initialization" in column.lower()
        ):
            output[column] = output[column].astype(
                "string"
            )

    output.to_csv(
        output_paths[key],
        index=False,
    )

protocol = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "estimator_scopes": scope_registry[
        [
            "scope_id",
            "scope_type",
            "applicable_decision_rule",
        ]
    ].to_dict(orient="records"),
    "minimum_training_dates": MIN_TRAINING_DATES,
    "minimum_calibration_dates": (
        MIN_CALIBRATION_DATES
    ),
    "fold_training_rule": (
        "training date strictly before first validation "
        "date and HKO label available by earliest applicable "
        "validation decision"
    ),
    "model_specific_holdout_freeze_rule": (
        "earliest applicable decision time in the "
        "22-31 May locked holdout"
    ),
    "selection_support_rule": (
        "development OOF row and realised label available "
        "by the model-specific holdout freeze"
    ),
    "common_candidate_oof_support_intersection_pending": True,
    "final_fit_rule": (
        "pre-holdout residual observation admitted by the "
        "model-specific holdout freeze"
    ),
    "holdout_protocol": (
        "fit and selection frozen before first applicable "
        "holdout decision; no holdout label used"
    ),
    "external_primary_protocol": (
        "transfer identical pre-holdout fitted estimator "
        "to June without refit on holdout labels"
    ),
    "external_operational_refit_sensitivity_deferred": True,
    "holdout_used_for_model_selection": False,
    "external_test_used_for_model_selection": False,
    "model_specification_selected": False,
    "score_hierarchy_selected": False,
    "calibrator_variant_selected": False,
    "trading_threshold_selected": False,
    "final_modelling_split_assigned": True,
    "probability_bridge_retained": False,
    "market_information_used_for_support_design": False,
}

protocol_path = OUT_DIR / "18uB_protocol.json"
protocol_path.write_text(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

source_inventory = pd.DataFrame(
    [
        {
            "input_role": "18uA_date_rule_assignment",
            "path": str(
                ASSIGNMENT_PATH.relative_to(ROOT)
            ),
            "rows": len(assignment),
            "sha256": sha256_file(ASSIGNMENT_PATH),
        },
        {
            "input_role": "18uA_date_partition",
            "path": str(
                DATE_PARTITION_PATH.relative_to(ROOT)
            ),
            "rows": len(date_partition),
            "sha256": sha256_file(
                DATE_PARTITION_PATH
            ),
        },
        {
            "input_role": "18uA_summary",
            "path": str(
                U_SUMMARY_PATH.relative_to(ROOT)
            ),
            "rows": 1,
            "sha256": sha256_file(U_SUMMARY_PATH),
        },
        {
            "input_role": "18tA_availability_panel",
            "path": str(
                AVAILABILITY_PATH.relative_to(ROOT)
            ),
            "rows": len(availability),
            "sha256": sha256_file(
                AVAILABILITY_PATH
            ),
        },
        {
            "input_role": "18tB_residual_observation_panel",
            "path": str(
                RESIDUAL_PATH.relative_to(ROOT)
            ),
            "rows": len(residuals),
            "sha256": sha256_file(RESIDUAL_PATH),
        },
        {
            "input_role": "18tB_same_rule_history_pairs",
            "path": str(
                SAME_RULE_PAIRS_PATH.relative_to(ROOT)
            ),
            "rows": len(same_rule_pairs),
            "sha256": sha256_file(
                SAME_RULE_PAIRS_PATH
            ),
        },
    ]
)
source_inventory_path = (
    OUT_DIR / "18uB_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "estimator_scopes": int(len(scope_registry)),
    "scope_fold_rows": int(len(fold_plan)),
    "minimum_training_dates": MIN_TRAINING_DATES,
    "minimum_calibration_dates": (
        MIN_CALIBRATION_DATES
    ),
    "fold_training_membership_rows": int(
        len(fold_training_membership)
    ),
    "fold_validation_membership_rows": int(
        len(fold_validation_membership)
    ),
    "oof_selection_support_rows_all_scopes": int(
        len(oof_selection_support)
    ),
    "freeze_admissible_selection_rows_all_scopes": int(
        oof_selection_support[
            "freeze_admissible_for_selection"
        ].sum()
    ),
    "final_fit_membership_rows_all_scopes": int(
        len(final_fit_membership)
    ),
    "evaluation_support_rows_all_scopes": int(
        len(evaluation_support)
    ),
    "pooled_scope": (
        freeze_summary.loc[
            freeze_summary["scope_id"].eq(
                "pooled_all_rules"
            )
        ].iloc[0].to_dict()
    ),
    "by_scope": freeze_summary.to_dict(
        orient="records"
    ),
    "holdout_labels_used_for_fitting": False,
    "external_labels_used_for_fitting": False,
    "external_refit_on_holdout_labels": False,
    "common_candidate_oof_support_intersection_pending": True,
    "model_specification_selected": False,
    "score_hierarchy_selected": False,
    "final_modelling_split_assigned": True,
    "probability_bridge_retained": False,
    "market_information_used_for_support_design": False,
    "issue_rows": 0,
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(len(integrity)),
}

def json_default(value: Any) -> Any:
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(
        f"Not JSON serialisable: {type(value)}"
    )

summary_path = OUT_DIR / "18uB_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
        default=json_default,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "revision": "v1",
}
environment_path = OUT_DIR / "18uB_environment.json"
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 18uB model-specific freeze support",
    "",
    "**PASS**",
    "",
    "## Model-specific holdout freezes",
    "",
    (
        "| Scope | Freeze UTC | OOF rows | Selection rows | "
        "Fit rows | Holdout rows | External rows |"
    ),
    "|---|---|---:|---:|---:|---:|---:|",
]

for row in freeze_summary.itertuples(index=False):
    report_lines.append(
        f"| {row.scope_id} | "
        f"{row.model_specific_holdout_freeze_utc} | "
        f"{int(row.development_oof_rows)} | "
        f"{int(row.freeze_admissible_selection_rows)} | "
        f"{int(row.final_fit_rows)} | "
        f"{int(row.internal_holdout_rows)} | "
        f"{int(row.external_test_rows)} |"
    )

report_lines.extend(
    [
        "",
        "## Frozen safeguards",
        "",
        (
            "- Every fold training date strictly precedes "
            "its first validation date."
        ),
        (
            "- Every fold training label is available by "
            "the earliest applicable validation decision."
        ),
        (
            "- Holdout and external labels are excluded from "
            "selection, fitting and calibration support."
        ),
        (
            "- June receives the identical pre-holdout fit "
            "in the primary external evaluation."
        ),
        "",
        "## Deferred choices",
        "",
        (
            "GP kernel, tree specification, feature family, "
            "score hierarchy, calibrator and trading threshold "
            "remain unselected."
        ),
    ]
)

report_path = (
    REPORT_DIR
    / "18uB_model_specific_freeze_support_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18uB_sha256_manifest.csv":
            continue

        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = OUT_DIR / "18uB_sha256_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2, default=json_default))
print("18uB model-specific freeze support release: PASS")

{
  "step": "18uB",
  "generated_at_utc": "2026-07-21T21:54:39.156938+00:00",
  "verdict": "PASS",
  "estimator_scopes": 5,
  "scope_fold_rows": 20,
  "minimum_training_dates": 16,
  "minimum_calibration_dates": 8,
  "fold_training_membership_rows": 924,
  "fold_validation_membership_rows": 288,
  "oof_selection_support_rows_all_scopes": 288,
  "freeze_admissible_selection_rows_all_scopes": 274,
  "final_fit_membership_rows_all_scopes": 418,
  "evaluation_support_rows_all_scopes": 318,
  "pooled_scope": {
    "scope_id": "pooled_all_rules",
    "scope_type": "POOLED",
    "applicable_decision_rule": "",
    "model_specific_holdout_freeze_utc": "2026-05-20T16:00:00+00:00",
    "development_oof_rows": 144,
    "development_oof_dates": 38,
    "freeze_admissible_selection_rows": 136,
    "freeze_admissible_selection_dates": 36,
    "latest_selection_label_date": "2026-05-19T00:00:00",
    "final_fit_rows": 208,
    "final_fit_dates": 60,
    "latest_final_fit_label_date": "2026-05-19T00:0